In [ ]:
"""
SeizeIT2 Dataset Download and Preparation

Ce script télécharge et prépare le dataset SeizeIT2 pour la détection d'épilepsie.

Dataset: SeizeIT2 - Wearable Dataset Of Patients With Focal Epilepsy
GitHub: https://github.com/biomedepi/seizeit2

Améliorations:
- Téléchargement automatique depuis GitHub
- Prétraitement adapté aux capteurs portables
- Segmentation temporelle optimisée
- Équilibrage des classes avec SMOTE
"""

In [ ]:
!pip install tensorflow keras scipy scikit-learn imbalanced-learn
!pip install requests pandas matplotlib seaborn tqdm numpy pywavelets neurokit2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from imblearn.over_sampling import SMOTE
import requests
import zipfile
from io import BytesIO
import seaborn as sns
from scipy import signal
import pywt
import neurokit2 as nk

# Configuration
BASE_PATH = '/content/drive/MyDrive/SeizeIT2_Project/'
RAW_DATA_PATH = BASE_PATH + 'raw_data/'
PROCESSED_DATA_PATH = BASE_PATH + 'processed_data/'
DATA_SPLIT_PATH = BASE_PATH + 'data_split/'
PLOTS_PATH = BASE_PATH + 'plots/'

# Créer la structure de dossiers
for path in [BASE_PATH, RAW_DATA_PATH, PROCESSED_DATA_PATH, DATA_SPLIT_PATH, PLOTS_PATH]:
    os.makedirs(path, exist_ok=True)

print("🔧 Structure de dossiers créée!")

In [ ]:
class SeizeIT2Processor:
    """
    Classe pour traiter les données SeizeIT2 avec améliorations
    """
    
    def __init__(self, sampling_rate=250, window_size=4, overlap=0.5):
        self.sampling_rate = sampling_rate
        self.window_size = window_size
        self.overlap = overlap
        self.window_samples = int(window_size * sampling_rate)
        self.scaler = RobustScaler()
        
    def download_seizeit2_dataset(self):
        """Télécharge le dataset SeizeIT2 depuis GitHub"""
        print("📥 Téléchargement du dataset SeizeIT2...")
        
        github_url = "https://github.com/biomedepi/seizeit2/archive/main.zip"
        
        try:
            response = requests.get(github_url, stream=True)
            response.raise_for_status()
            
            with zipfile.ZipFile(BytesIO(response.content)) as zip_ref:
                zip_ref.extractall(RAW_DATA_PATH)
            
            print("✅ Dataset SeizeIT2 téléchargé avec succès!")
            self.list_downloaded_files()
            
        except Exception as e:
            print(f"❌ Erreur lors du téléchargement: {e}")
            print("🎲 Génération de données simulées...")
            return self.generate_simulated_data()
    
    def list_downloaded_files(self):
        """Liste les fichiers téléchargés"""
        print("\n📂 Fichiers téléchargés:")
        for root, dirs, files in os.walk(RAW_DATA_PATH):
            level = root.replace(RAW_DATA_PATH, '').count(os.sep)
            indent = ' ' * 2 * level
            print(f"{indent}{os.path.basename(root)}/")
            subindent = ' ' * 2 * (level + 1)
            for file in files[:5]:
                print(f"{subindent}{file}")
            if len(files) > 5:
                print(f"{subindent}... et {len(files)-5} autres fichiers")
    
    def generate_simulated_data(self):
        """Génère des données simulées réalistes pour SeizeIT2"""
        print("🎲 Génération de données simulées SeizeIT2...")
        
        # Paramètres de simulation
        n_patients = 20
        n_sessions_per_patient = 10
        session_duration = 300  # 5 minutes
        n_samples_per_session = session_duration * self.sampling_rate
        n_sensors = 6  # ACC_X, ACC_Y, ACC_Z, GYRO_X, GYRO_Y, GYRO_Z
        
        all_data = []
        all_labels = []
        
        print(f"   👥 {n_patients} patients")
        print(f"   📊 {n_sessions_per_patient} sessions par patient")
        print(f"   ⏱️ {session_duration}s par session")
        print(f"   📡 {n_sensors} capteurs")
        
        for patient_id in tqdm(range(n_patients), desc="Génération patients"):
            for session_id in range(n_sessions_per_patient):
                # Générer des données de capteurs réalistes
                data = self.generate_realistic_sensor_data(
                    n_samples_per_session, n_sensors, patient_id
                )
                
                # Générer des labels avec crises occasionnelles
                labels = self.generate_seizure_labels(n_samples_per_session, session_id)
                
                all_data.append(data)
                all_labels.extend(labels)
        
        X = np.vstack(all_data)
        y = np.array(all_labels)
        
        print(f"✅ Données simulées générées: {X.shape}")
        print(f"   Distribution: Normal={np.sum(y==0)}, Crise={np.sum(y==1)}")
        
        return X, y
    
    def generate_realistic_sensor_data(self, n_samples, n_sensors, patient_id):
        """Génère des données de capteurs réalistes"""
        # Paramètres de base pour chaque capteur
        base_params = {
            0: {'mean': 0.2, 'std': 0.8, 'freq': 1.5},    # ACC_X
            1: {'mean': -0.1, 'std': 0.6, 'freq': 1.2},   # ACC_Y  
            2: {'mean': 9.8, 'std': 1.0, 'freq': 0.8},    # ACC_Z
            3: {'mean': 0.0, 'std': 0.3, 'freq': 2.0},    # GYRO_X
            4: {'mean': 0.0, 'std': 0.25, 'freq': 1.8},   # GYRO_Y
            5: {'mean': 0.0, 'std': 0.2, 'freq': 2.2}     # GYRO_Z
        }
        
        time = np.linspace(0, n_samples/self.sampling_rate, n_samples)
        data = np.zeros((n_samples, n_sensors))
        
        for sensor in range(n_sensors):
            params = base_params[sensor]
            
            # Signal de base avec variation par patient
            patient_factor = 1 + 0.2 * np.sin(patient_id * np.pi / 10)
            
            signal_data = (
                params['mean'] * patient_factor +
                params['std'] * np.random.randn(n_samples) * 0.3 +
                0.5 * np.sin(2 * np.pi * params['freq'] * time) +
                0.2 * np.sin(2 * np.pi * params['freq'] * 3 * time) +
                0.1 * np.random.randn(n_samples)
            )
            
            data[:, sensor] = signal_data
        
        return data
    
    def generate_seizure_labels(self, n_samples, session_id):
        """Génère des labels avec des patterns de crises réalistes"""
        labels = np.zeros(n_samples)
        
        # 20% de chance d'avoir une crise dans une session
        if np.random.random() < 0.2:
            # Durée de crise: 30-120 secondes
            seizure_duration = np.random.randint(30, 121) * self.sampling_rate
            seizure_start = np.random.randint(0, max(1, n_samples - seizure_duration))
            seizure_end = min(seizure_start + seizure_duration, n_samples)
            
            labels[seizure_start:seizure_end] = 1
        
        return labels
    
    def apply_preprocessing(self, X):
        """Applique un prétraitement avancé aux signaux"""
        print("🔧 Prétraitement des signaux...")
        
        X_processed = np.zeros_like(X)
        
        for sensor in range(X.shape[1]):
            # Normalisation robuste
            X_processed[:, sensor] = self.scaler.fit_transform(
                X[:, sensor].reshape(-1, 1)
            ).flatten()
            
            # Filtrage passe-bande (0.5-50 Hz)
            X_processed[:, sensor] = self.apply_bandpass_filter(
                X_processed[:, sensor], 0.5, 50
            )
            
            # Débruitage par ondelettes
            X_processed[:, sensor] = self.apply_wavelet_denoising(
                X_processed[:, sensor]
            )
        
        return X_processed
    
    def apply_bandpass_filter(self, signal_data, low_freq, high_freq):
        """Applique un filtre passe-bande"""
        from scipy.signal import butter, filtfilt
        
        nyquist = self.sampling_rate / 2
        low = low_freq / nyquist
        high = high_freq / nyquist
        
        if high >= 1.0:
            high = 0.99
        
        b, a = butter(4, [low, high], btype='band')
        return filtfilt(b, a, signal_data)
    
    def apply_wavelet_denoising(self, signal_data):
        """Applique un débruitage par ondelettes"""
        # Décomposition en ondelettes
        coeffs = pywt.wavedec(signal_data, 'db4', level=5)
        
        # Seuillage doux
        threshold = 0.04
        coeffs_thresh = list(coeffs)
        coeffs_thresh[1:] = [pywt.threshold(c, threshold, mode='soft') for c in coeffs_thresh[1:]]
        
        # Reconstruction
        return pywt.waverec(coeffs_thresh, 'db4')
    
    def segment_data(self, X, y):
        """Segmente les données en fenêtres temporelles"""
        print("✂️ Segmentation des données...")
        
        n_samples = len(X)
        step = int(self.window_samples * (1 - self.overlap))
        
        windows = []
        labels = []
        
        for start in tqdm(range(0, n_samples - self.window_samples, step), 
                         desc="Segmentation"):
            end = start + self.window_samples
            
            # Extraire la fenêtre
            window = X[start:end]
            
            # Déterminer le label (majorité dans la fenêtre)
            window_labels = y[start:end]
            label = 1 if np.mean(window_labels) > 0.5 else 0
            
            windows.append(window.T)  # Transposer pour avoir (n_sensors, n_timepoints)
            labels.append(label)
        
        return np.array(windows), np.array(labels)
    
    def balance_dataset(self, X, y):
        """Équilibre le dataset avec SMOTE"""
        print("⚖️ Équilibrage des données avec SMOTE...")
        
        # Reshape pour SMOTE
        n_samples, n_sensors, n_timepoints = X.shape
        X_reshaped = X.reshape(n_samples, -1)
        
        # Appliquer SMOTE
        smote = SMOTE(random_state=42, k_neighbors=3)
        X_balanced, y_balanced = smote.fit_resample(X_reshaped, y)
        
        # Reshape back
        X_balanced = X_balanced.reshape(-1, n_sensors, n_timepoints)
        
        print(f"✅ Équilibrage terminé!")
        print(f"   Nouvelles dimensions: {X_balanced.shape}")
        print(f"   Classes équilibrées: {np.bincount(y_balanced)}")
        
        return X_balanced, y_balanced
    
    def create_data_splits(self, X, y, test_size=0.2, val_size=0.2):
        """Divise les données en train/validation/test"""
        print("📊 Division des données...")
        
        # Premier split: train+val vs test
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        
        # Deuxième split: train vs val
        val_size_adjusted = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=val_size_adjusted, random_state=42, stratify=y_temp
        )
        
        print(f"   Train: {X_train.shape}")
        print(f"   Validation: {X_val.shape}")
        print(f"   Test: {X_test.shape}")
        
        return X_train, X_val, X_test, y_train, y_val, y_test
    
    def save_processed_data(self, X_train, X_val, X_test, y_train, y_val, y_test):
        """Sauvegarde les données traitées"""
        print("💾 Sauvegarde des données...")
        
        np.save(DATA_SPLIT_PATH + 'X_train_raw.npy', X_train)
        np.save(DATA_SPLIT_PATH + 'X_val_raw.npy', X_val)
        np.save(DATA_SPLIT_PATH + 'X_test_raw.npy', X_test)
        np.save(DATA_SPLIT_PATH + 'y_train.npy', y_train)
        np.save(DATA_SPLIT_PATH + 'y_val.npy', y_val)
        np.save(DATA_SPLIT_PATH + 'y_test.npy', y_test)
        
        print(f"✅ Données sauvegardées dans {DATA_SPLIT_PATH}")
    
    def plot_data_overview(self, X, y):
        """Affiche un aperçu des données"""
        print("📈 Génération des visualisations...")
        
        fig, axes = plt.subplots(3, 2, figsize=(15, 12))
        
        # Distribution des classes
        axes[0, 0].bar(['Normal', 'Crise'], [np.sum(y == 0), np.sum(y == 1)])
        axes[0, 0].set_title('Distribution des Classes')
        axes[0, 0].set_ylabel('Nombre d\'échantillons')
        
        # Exemple de signaux par classe
        normal_idx = np.where(y == 0)[0][0]
        seizure_idx = np.where(y == 1)[0][0]
        
        for i in range(6):  # 6 capteurs
            if i < 3:
                ax = axes[1, i//3] if i < 2 else axes[2, 0]
            else:
                ax = axes[2, (i-3)//2] if i < 5 else axes[2, 1]
            
            if i < X.shape[1]:
                ax.plot(X[normal_idx, i, :], label='Normal', alpha=0.7)
                ax.plot(X[seizure_idx, i, :], label='Crise', alpha=0.7)
                ax.set_title(f'Capteur {i+1}')
                ax.set_xlabel('Échantillons')
                ax.set_ylabel('Amplitude')
                ax.legend()
        
        # Distribution des amplitudes
        axes[0, 1].hist(X[y==0].flatten(), bins=50, alpha=0.5, label='Normal', density=True)
        axes[0, 1].hist(X[y==1].flatten(), bins=50, alpha=0.5, label='Crise', density=True)
        axes[0, 1].set_title('Distribution des Amplitudes')
        axes[0, 1].set_xlabel('Amplitude')
        axes[0, 1].set_ylabel('Densité')
        axes[0, 1].legend()
        
        plt.tight_layout()
        plt.savefig(PLOTS_PATH + 'data_overview.png', dpi=150)
        plt.show()

In [ ]:
def main():
    """Fonction principale"""
    print("🚀 Démarrage du traitement SeizeIT2")
    print("=" * 60)
    
    # Initialiser le processeur
    processor = SeizeIT2Processor(
        sampling_rate=250,  # SeizeIT2 utilise 250Hz
        window_size=4,      # Fenêtres de 4 secondes
        overlap=0.5         # 50% de chevauchement
    )
    
    # 1. Télécharger/Générer le dataset
    X, y = processor.download_seizeit2_dataset()
    
    # 2. Prétraitement
    X_processed = processor.apply_preprocessing(X)
    
    # 3. Segmentation
    X_segmented, y_segmented = processor.segment_data(X_processed, y)
    
    # 4. Équilibrage
    X_balanced, y_balanced = processor.balance_dataset(X_segmented, y_segmented)
    
    # 5. Division des données
    X_train, X_val, X_test, y_train, y_val, y_test = processor.create_data_splits(
        X_balanced, y_balanced
    )
    
    # 6. Sauvegarde
    processor.save_processed_data(X_train, X_val, X_test, y_train, y_val, y_test)
    
    # 7. Visualisations
    processor.plot_data_overview(X_train, y_train)
    
    print("\n✅ Préparation des données SeizeIT2 terminée!")
    print("📊 Résumé:")
    print(f"   • Dataset: SeizeIT2 (capteurs portables)")
    print(f"   • Échantillonnage: {processor.sampling_rate} Hz")
    print(f"   • Fenêtres: {processor.window_size}s avec {processor.overlap*100}% chevauchement")
    print(f"   • Train: {X_train.shape}")
    print(f"   • Validation: {X_val.shape}")
    print(f"   • Test: {X_test.shape}")
    print(f"   • Classes équilibrées: {np.bincount(y_train)}")
    print("\n🔄 Prochaine étape: Transformation en images")

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
try:
    X_train = np.load(DATA_SPLIT_PATH + 'X_train_raw.npy')
    X_val = np.load(DATA_SPLIT_PATH + 'X_val_raw.npy')
    X_test = np.load(DATA_SPLIT_PATH + 'X_test_raw.npy')
    y_train = np.load(DATA_SPLIT_PATH + 'y_train.npy')
    y_val = np.load(DATA_SPLIT_PATH + 'y_val.npy')
    y_test = np.load(DATA_SPLIT_PATH + 'y_test.npy')
    
    print("✅ Données chargées avec succès:")
    print(f"   X_train: {X_train.shape}")
    print(f"   X_val: {X_val.shape}")
    print(f"   X_test: {X_test.shape}")
    print(f"   y_train: {y_train.shape} - Distribution: {np.bincount(y_train)}")
    print(f"   y_val: {y_val.shape} - Distribution: {np.bincount(y_val)}")
    print(f"   y_test: {y_test.shape} - Distribution: {np.bincount(y_test)}")
    
except FileNotFoundError as e:
    print(f"❌ Erreur de chargement: {e}")
    print("🔄 Relancez la cellule principale pour générer les données")